<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS11/datasetPBL/TugasPraktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

tugas 1

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Load dan gabungkan dataset milikmu
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')
df = pd.merge(df_reviews, df_cafes, on='cafe_id', how='left', suffixes=('_review', '_cafe'))

# 2. Ekstraksi fitur numerik + teks (TF-IDF) & standardisasi
tfidf_20 = TfidfVectorizer(max_features=20, ngram_range=(1, 2))
X_txt20 = tfidf_20.fit_transform((df['name'] + " " + df['text'].astype(str))).toarray()
X_num = df[['rating_review', 'rating_cafe', 'review_count', 'latitude', 'longitude']].fillna(0).values

X = StandardScaler().fit_transform(np.hstack([X_num, X_txt20]))
y = (df['categories'] == "['Kedai Kopi']").astype(int).values

# 3. Pengujian kombinasi split (70:30 dan 80:20) serta kernel (linear, poly, rbf)
splits = [(0.30, '70:30'), (0.20, '80:20')]
kernels = ['linear', 'poly', 'rbf']

hasil_tugas1 = []

for test_size, nama_split in splits:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )
    for kernel in kernels:
        model = SVC(kernel=kernel, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        hasil_tugas1.append({
            'Rasio Split': nama_split,
            'Kernel': kernel.upper(),
            'Akurasi': round(acc, 4),
            'Akurasi (%)': f"{acc * 100:.2f}%"
        })

# 4. Tabulasi hasil performansi
df_hasil_t1 = pd.DataFrame(hasil_tugas1)
print(df_hasil_t1.to_string(index=False))

Rasio Split Kernel  Akurasi Akurasi (%)
      70:30 LINEAR   0.7997      79.97%
      70:30   POLY   0.8640      86.40%
      70:30    RBF   0.8714      87.14%
      80:20 LINEAR   0.8004      80.04%
      80:20   POLY   0.8789      87.89%
      80:20    RBF   0.8924      89.24%


tugas 2

In [2]:
import cv2
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# 1. Siapkan data citra representasi dari dataset milikmu (melanjutkan Praktikum 5)
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')
df = pd.merge(df_reviews, df_cafes, on='cafe_id', how='left', suffixes=('_review', '_cafe'))

tfidf_300 = TfidfVectorizer(max_features=300, ngram_range=(1, 2))
X_tfidf300 = tfidf_300.fit_transform((df['name'] + " " + df['text'].astype(str))).toarray()
r_norm = (df['rating_cafe'] / 5.0).values[:, None]
rc_norm = (df['review_count'] / df['review_count'].max()).values[:, None]

feats = np.hstack([X_tfidf300[:, :298], r_norm, rc_norm])
labels_str = np.where(df['categories'] == "['Kedai Kopi']", 'day', 'night')

rng = np.random.default_rng(42)
dataset_img = []
for i in range(400):
    base = (feats[i].reshape(10, 10, 3) * 180).astype(np.float32)
    jitter = rng.normal(0, 18)
    if labels_str[i] == 'day':
        base = np.clip(base + 115 + jitter, 0, 255)
    else:
        base = np.clip(base + 65 + jitter, 0, 255)
    label_num = 1 if labels_str[i] == 'day' else 0
    dataset_img.append((base.astype(np.uint8), label_num))

# 2. Fungsi ekstraksi fitur Histogram Warna HSV (8x8x8 = 512 fitur)
def extract_hsv_histogram(image, bins=(8, 8, 8)):
    std_img = cv2.resize(image, (1100, 600))
    hsv = cv2.cvtColor(std_img, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

X_hist = np.array([extract_hsv_histogram(img) for img, _ in dataset_img])
y_hist = np.array([lbl for _, lbl in dataset_img])

# 3. Split data dengan rasio 80:20
X_train, X_test, y_train, y_test = train_test_split(
    X_hist, y_hist, test_size=0.20, random_state=42, stratify=y_hist
)

# 4. Evaluasi SVM Kernel RBF Baseline (Default)
svm_default = SVC(kernel='rbf', random_state=42)
svm_default.fit(X_train, y_train)
acc_default = accuracy_score(y_test, svm_default.predict(X_test))

# 5. Hyperparameter Tuning Kernel RBF (C dan gamma) dengan GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 50, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01, 0.001]
}
grid_search = GridSearchCV(SVC(kernel='rbf', random_state=42), param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
acc_tuned = accuracy_score(y_test, y_pred_tuned)

print(f"Akurasi RBF Default (C=1.0, gamma='scale') : {acc_default:.4f} ({acc_default*100:.2f}%)")
print(f"Parameter Terbaik Hasil Tuning             : {grid_search.best_params_}")
print(f"Akurasi Terbaik Cross-Validation (Train)   : {grid_search.best_score_:.4f}")
print(f"Akurasi RBF Setelah Tuning (Test 80:20)    : {acc_tuned:.4f} ({acc_tuned*100:.2f}%)\n")
print(classification_report(y_test, y_pred_tuned, target_names=['Night / Lainnya (0)', 'Day / Kedai Kopi (1)']))

Akurasi RBF Default (C=1.0, gamma='scale') : 0.9125 (91.25%)
Parameter Terbaik Hasil Tuning             : {'C': 0.1, 'gamma': 'scale'}
Akurasi Terbaik Cross-Validation (Train)   : 0.9219
Akurasi RBF Setelah Tuning (Test 80:20)    : 0.9125 (91.25%)

                      precision    recall  f1-score   support

 Night / Lainnya (0)       0.91      0.96      0.93        50
Day / Kedai Kopi (1)       0.93      0.83      0.88        30

            accuracy                           0.91        80
           macro avg       0.92      0.90      0.90        80
        weighted avg       0.91      0.91      0.91        80

